# Preparación histórica de CSV
Extracto de la versión revisada del proyecto original, sin credenciales ni conexión a base de datos. No ejecutado en esta revisión: requiere los archivos brutos en `data/raw/`. Las salidas guardadas son históricas. La exportación se dirige a `data/reconstruidos/` para conservar los CSV aportados.

El filtro exacto de monto puede descartar diferencias de centavos. Antes de reutilizarlo, investigar la causa y definir una tolerancia justificada. El filtro cantidad >= 0 también excluye nulos; la etiqueta histórica de «cantidad negativa» no representa todos los descartes.


In [ ]:
from pathlib import Path
import pandas as pd
ROOT = Path.cwd()
if not (ROOT/'data').is_dir(): ROOT = ROOT.parent
orders = pd.read_csv(ROOT/'data/raw/rappiplus_orders_raw.csv')
catalog = pd.read_csv(ROOT/'data/raw/rappiplus_catalog.csv')
marketing = pd.read_csv(ROOT/'data/raw/rappiplus_marketing_spend.csv')


In [21]:
# ── ORDERS ──────────────────────────────────────────────────
 
print("=" * 60)
print("LIMPIEZA: ORDERS")
print("=" * 60)
 
# 1. Convertir fechas al formato correcto
orders["fecha_hora_pedido"] = pd.to_datetime(orders["fecha_hora_pedido"], errors="coerce")
print(f"Fechas nulas tras conversión: {orders['fecha_hora_pedido'].isna().sum()}")
 
# 2. Revisar variables numéricas (sin negativos ni ceros inválidos)
for col in ["cantidad", "precio_unitario", "monto_total"]:
    n_neg  = (orders[col] < 0).sum()
    n_zero = (orders[col] == 0).sum()
    print(f"  {col}: negativos={n_neg}, ceros={n_zero}")
 
# 3. Verificar consistencia de montos
# monto_total esperado = precio_unitario * cantidad - monto_descuento
orders = orders[orders["cantidad"] >= 0]
print(f"Filas eliminadas por cantidad negativa. Orders: {orders.shape[0]:,} filas")

orders["monto_esperado"] = (
    orders["precio_unitario"] * orders["cantidad"] - orders["monto_descuento"]
).round(2)
inconsistentes = (orders["monto_total"].round(2) != orders["monto_esperado"]).sum()
print(f"Filas con monto_total inconsistente: {inconsistentes}")
orders = orders[orders["monto_total"].round(2) == orders["monto_esperado"]]
print(f"Filas eliminadas por monto inconsistente. Orders: {orders.shape[0]:,} filas")
orders.drop(columns=["monto_esperado"], inplace=True)
 
# 4. Eliminar duplicados
dup_orders = orders.duplicated().sum()
print(f"Duplicados en orders: {dup_orders}")
orders.drop_duplicates(inplace=True)
 
# 5. Revisar variables categóricas
for col in ["pais", "dispositivo", "fuente_referencia", "categoria_producto"]:
    print(f"\n  Valores únicos en '{col}':")
    print("  ", orders[col].value_counts().to_dict())

orders["pais"] = orders["pais"].str.capitalize()
print(f"\nValores únicos en 'pais' tras normalización:")
print(orders["pais"].value_counts().to_dict())

print(f"\nOrders limpio: {orders.shape[0]:,} filas")
 

LIMPIEZA: ORDERS
Fechas nulas tras conversión: 0
  cantidad: negativos=4, ceros=0
  precio_unitario: negativos=0, ceros=0
  monto_total: negativos=4, ceros=0
Filas eliminadas por cantidad negativa. Orders: 25,046 filas
Filas con monto_total inconsistente: 6256
Filas eliminadas por monto inconsistente. Orders: 18,790 filas
Duplicados en orders: 77

  Valores únicos en 'pais':
   {'Colombia': 5609, 'Mexico': 5581, 'Argentina': 5433, 'mexico': 667, 'colombia': 605, 'argentina': 600}

  Valores únicos en 'dispositivo':
   {'desktop': 9532, 'mobile': 9165}

  Valores únicos en 'fuente_referencia':
   {'social': 6277, 'paid_search': 6231, 'organic': 6184}

  Valores únicos en 'categoria_producto':
   {'Moda': 6300, 'Hogar': 6251, 'Electronica': 6141}

Valores únicos en 'pais' tras normalización:
{'Mexico': 6248, 'Colombia': 6214, 'Argentina': 6033}

Orders limpio: 18,713 filas


In [22]:
# ── CATALOG ─────────────────────────────────────────────────
 
print("\n" + "=" * 60)
print("LIMPIEZA: CATALOG")
print("=" * 60)
 
# 1. No hay fechas en catalog
 
# 2. Revisar numéricos
n_neg_costo = (catalog["costo_unitario"] <= 0).sum()
print(f"Costos unitarios <= 0: {n_neg_costo}")
 
# 3. Duplicados
dup_catalog = catalog.duplicated().sum()
print(f"Duplicados en catalog: {dup_catalog}")
catalog.drop_duplicates(inplace=True)
 
# 4. Categorías
print("\nCategorías en catalog:")
print(catalog["categoria_producto"].value_counts().to_dict())
 
print(f"\nCatalog limpio: {catalog.shape[0]} filas")
 


LIMPIEZA: CATALOG
Costos unitarios <= 0: 0
Duplicados en catalog: 0

Categorías en catalog:
{'Electrónica': 3, 'Hogar': 2, 'Moda': 2}

Catalog limpio: 7 filas


In [23]:
# ── MARKETING ────────────────────────────────────────────────
 
print("\n" + "=" * 60)
print("LIMPIEZA: MARKETING")
print("=" * 60)
 
# 1. Convertir fechas
marketing["fecha"] = pd.to_datetime(marketing["fecha"], errors="coerce")
print(f"Fechas nulas tras conversión: {marketing['fecha'].isna().sum()}")
 
# 2. Revisar numéricos
n_neg_gasto = (marketing["gasto"] < 0).sum()
n_zero_gasto = (marketing["gasto"] == 0).sum()
print(f"Gasto negativo: {n_neg_gasto}, gasto cero: {n_zero_gasto}")
 
# 3. Duplicados
dup_marketing = marketing.duplicated().sum()
print(f"Duplicados en marketing: {dup_marketing}")
marketing.drop_duplicates(inplace=True)
 
# 4. Categorías
for col in ["pais", "canal"]:
    print(f"\nValores únicos en '{col}':")
    print(" ", marketing[col].value_counts().to_dict())
 
print(f"\nMarketing limpio: {marketing.shape[0]:,} filas")


LIMPIEZA: MARKETING
Fechas nulas tras conversión: 0
Gasto negativo: 0, gasto cero: 0
Duplicados en marketing: 0

Valores únicos en 'pais':
  {'Colombia': 540, 'Mexico': 540, 'Argentina': 540}

Valores únicos en 'canal':
  {'paid_search': 507, 'organic': 506, 'social': 506}

Marketing limpio: 1,620 filas


In [ ]:
out = ROOT/'data/reconstruidos'
out.mkdir(exist_ok=True)
orders.to_csv(out/'orders_clean.csv',index=False)
catalog.to_csv(out/'catalog_clean.csv',index=False)
marketing.to_csv(out/'marketing_clean.csv',index=False)
